# Day 76 — Agentic RAG

A naive RAG always searches. **Agentic RAG** lets the model decide: search when it needs
facts, answer directly when it doesn't. Retrieval stays offline; the decision + answer use
the model. > Needs a provider.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
import hashlib, math
from shared.llm import chat

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

DOCS = {"d1": "The office Wi-Fi password is 'sunflower42'."}
EMB = {k: cheap_embed(v) for k, v in DOCS.items()}

def search(query):
    qe = cheap_embed(query)
    return DOCS[max(EMB.items(), key=lambda kv: cosine(qe, kv[1]))[0]]

def agentic_rag(question):
    decision = chat([{"role": "system", "content":
        "Reply ONLY 'SEARCH' if internal/company facts are needed, else 'DIRECT'."},
        {"role": "user", "content": question}], temperature=0)
    # TODO: if 'SEARCH' in decision -> answer using search(question) as context;
    #       else answer the question directly with chat(). Return the answer text.
    raise NotImplementedError

# print(agentic_rag("What is the office Wi-Fi password?"))
# print(agentic_rag("What is 2 + 2?"))

## 🔒 Solution

In [ ]:
import hashlib, math
from shared.llm import chat

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

DOCS = {"d1": "The office Wi-Fi password is 'sunflower42'."}
EMB = {k: cheap_embed(v) for k, v in DOCS.items()}

def search(query):
    qe = cheap_embed(query)
    return DOCS[max(EMB.items(), key=lambda kv: cosine(qe, kv[1]))[0]]

def agentic_rag(question):
    decision = chat([{"role": "system", "content":
        "Reply ONLY 'SEARCH' if internal/company facts are needed, else 'DIRECT'."},
        {"role": "user", "content": question}], temperature=0)
    if "SEARCH" in decision.upper():
        ctx = search(question)
        return chat([{"role": "system", "content": "Answer using ONLY the context."},
                     {"role": "user", "content": f"Q: {question}\nContext: {ctx}"}], temperature=0)
    return chat([{"role": "user", "content": question}], temperature=0)

print(agentic_rag("What is the office Wi-Fi password?"))
print(agentic_rag("What is 2 + 2?"))